# Fadhma-300M → Tarifit Transfer Fine-Tuning on the Improved Corpus

**Goal:** fine-tune `agbalu/Fadhma-300M` on the improved Tarifit training corpus.

**Design:** new 38-token Tarifit CTC head; improved training corpus; cleaned 128-segment validation set; greedy CTC decoding; no external LM; CER used as the main selection metric.

> If the improved corpus keeps the same alphabet, reuse `mms_tokenizer_v1_1`. Only change the tokenizer path if the target vocabulary itself changes.

In [ ]:
# Cell 1 — Mount Google Drive and check the GPU

from google.colab import drive
drive.mount("/content/drive")

import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1), "GB")
else:
    print("GPU is not available. Fine-tuning should be postponed until a GPU is available.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Device: cuda
GPU: Tesla T4
GPU memory: 14.6 GB


In [ ]:
# Cell 2 — Install the required packages

!pip -q install -U transformers accelerate datasets jiwer soundfile tqdm
!pip -q install "pandas==2.2.3"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 86.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 45.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 87.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 87.8 MB/s eta 0:00:00


In [ ]:
# Cell 3 — Define the improved-corpus experiment paths

from pathlib import Path

PROJECT_ROOT = Path("/content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm")

# Change this folder name if you save the corrected corpus somewhere else.
IMPROVED_TRAIN_DIR = PROJECT_ROOT / "data" / "processed" / "mms_corpus_v1_1" / "train"
Exists: False
Validation exists: True
Tokenizer exists: True

[5]
2 s
Pasos siguientes:
[ ]
31 s

# Cell 5 — Load the Tarifit tokenizer and inspect corrected references

from transformers import Wav2Vec2CTCTokenizer

tokenizer = Wav2Vec2CTCTokenizer(
    vocab_file=str(TARIFIT_VOCAB_PATH),
    unk_token="[UNK]",
    pad_token="[PAD]",
    word_delimiter_token="|"
)

print("Tarifit vocabulary size:", len(tokenizer))
print("PAD / CTC blank ID:", tokenizer.pad_token_id)

for i in range(min(5, len(train_ds))):
    ref = tokenizer.decode(train_ds[i]["labels"], group_tokens=False)
    print(f"{i}: {ref}")

[ ]
0 s

# Cell 6 — Inspect improved corpus statistics

import numpy as np

train_durations = np.array([x["input_length"] / 16000 for x in train_ds])
train_label_lengths = np.array([len(x["labels"]) for x in train_ds])

print("Audio duration — min/max/mean:", round(train_durations.min(),2), round(train_durations.max(),2), round(train_durations.mean(),2))
print("Label length — min/max/mean:", int(train_label_lengths.min()), int(train_label_lengths.max()), round(train_label_lengths.mean(),2))

[ ]
0 s

# Cell 7 — Build the Fadhma feature extractor with the Tarifit tokenizer

from transformers import AutoFeatureExtractor, Wav2Vec2Processor

feature_extractor = AutoFeatureExtractor.from_pretrained(FADHMA_ID)
processor = Wav2Vec2Processor(feature_extractor=feature_extractor, tokenizer=tokenizer)

print("Fadhma base model:", FADHMA_ID)
print("Sampling rate:", feature_extractor.sampling_rate)
print("Audio normalization:", feature_extractor.do_normalize)
print("Tarifit output vocabulary:", len(tokenizer))

[ ]
0 s

# Cell 8 — Load Fadhma-300M with a new Tarifit CTC output head

from transformers import Wav2Vec2ForCTC

model = Wav2Vec2ForCTC.from_pretrained(
    FADHMA_ID,
    vocab_size=len(tokenizer),
    pad_token_id=tokenizer.pad_token_id,
    ctc_loss_reduction="mean",
    ctc_zero_infinity=True,
    ignore_mismatched_sizes=True,
)

model.freeze_feature_encoder()
model.gradient_checkpointing_enable()
model.config.mask_time_prob = 0.05
model.config.layerdrop = 0.0
model = model.to(device)

print("Starting model:", FADHMA_ID)
print("Total parameters:", f"{model.num_parameters():,}")
print("Trainable parameters:", f"{sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
print("Tarifit output vocabulary:", model.config.vocab_size)
print("CTC head:", model.lm_head)

[ ]
31 s

# Cell 9 — Verify CTC feasibility for improved training and validation

import torch

def minimum_ctc_frames(labels):
    repeats = sum(labels[i] == labels[i-1] for i in range(1, len(labels)))
    return len(labels) + repeats

def check_ctc_feasibility(dataset, name):
    invalid = []
    for i, example in enumerate(dataset):
        output_frames = int(model._get_feat_extract_output_lengths(torch.tensor(example["input_length"])).item())
        minimum_frames = minimum_ctc_frames(example["labels"])
        if minimum_frames > output_frames:
            invalid.append({
                "index": i,
                "duration_seconds": example["input_length"] / 16000,
                "label_length": len(example["labels"]),
                "minimum_ctc_frames": minimum_frames,
                "available_output_frames": output_frames,
            })
    print(name, "— total:", len(dataset), "valid:", len(dataset)-len(invalid), "invalid:", len(invalid))
    return invalid

bad_train = check_ctc_feasibility(train_ds, "IMPROVED TRAIN")
bad_val = check_ctc_feasibility(val_clean_ds, "CLEAN VALIDATION")

if bad_train:
    print("First invalid training examples:")
    for item in bad_train[:10]:
        print(item)

assert len(bad_val) == 0

[ ]
31 s

# Cell 10 — Remove newly detected CTC-infeasible training examples if needed

if len(bad_train) == 0:
    train_clean_ds = train_ds
    print("No improved-training examples need removal.")
else:
    bad_train_indices = {x["index"] for x in bad_train}
    good_train_indices = [i for i in range(len(train_ds)) if i not in bad_train_indices]
    train_clean_ds = train_ds.select(good_train_indices)
    print("Removed:", len(bad_train_indices))
    print("Final training examples:", len(train_clean_ds))

[ ]
31 s

# Cell 11 — Create the dynamic-padding CTC data collator

from dataclasses import dataclass
from typing import Union

@dataclass
class DataCollatorCTCWithPadding:
    processor: any
    padding: Union[bool, str] = True

    def __call__(self, features):
        input_features = [{"input_values": x["input_values"]} for x in features]
        label_features = [{"input_ids": x["labels"]} for x in features]

        batch = self.processor.pad(input_features, padding=self.padding, return_tensors="pt")
        labels_batch = self.processor.tokenizer.pad(label_features, padding=self.padding, return_tensors="pt")

        batch["labels"] = labels_batch["input_ids"].masked_fill(
            labels_batch["attention_mask"].ne(1), -100
        )
        return batch

data_collator = DataCollatorCTCWithPadding(processor=processor)

sample_batch = data_collator([train_clean_ds[0], train_clean_ds[1]])
print("Input batch shape:", sample_batch["input_values"].shape)
print("Label batch shape:", sample_batch["labels"].shape)

[ ]
31 s

# Cell 12 — Define the common evaluation normalization and WER/CER metrics

import re
import unicodedata
import numpy as np
from jiwer import wer, cer

def eval_normalize(text):
    text = unicodedata.normalize("NFC", str(text))
    text = text.lower().strip()
    text = re.sub(r"\s+", " ", text)
    return text

def compute_metrics(pred):
    pred_ids = np.argmax(pred.predictions, axis=-1)
    label_ids = pred.label_ids.copy()
    label_ids[label_ids == -100] = tokenizer.pad_token_id

    pred_str = processor.batch_decode(pred_ids)
    label_str = tokenizer.batch_decode(label_ids, group_tokens=False)

    pred_str = [eval_normalize(x) for x in pred_str]
    label_str = [eval_normalize(x) for x in label_str]

    return {"wer": wer(label_str, pred_str), "cer": cer(label_str, pred_str)}

print("Metrics ready.")

[ ]
31 s

# Cell 13 — Recover the previous OmniASR optimization settings

import torch

training_args_files = sorted(OMNI_REFERENCE_DIR.glob("checkpoint-*/training_args.bin"))

if not training_args_files:
    raise FileNotFoundError("No OmniASR training_args.bin found. Check OMNI_REFERENCE_DIR in Cell 3.")

REFERENCE_ARGS_FILE = training_args_files[-1]
omni_args = torch.load(REFERENCE_ARGS_FILE, map_location="cpu", weights_only=False)

fields = [
    "learning_rate", "per_device_train_batch_size", "per_device_eval_batch_size",
    "gradient_accumulation_steps", "num_train_epochs", "warmup_steps",
    "weight_decay", "seed"
]

print("Reference args:", REFERENCE_ARGS_FILE)
for field in fields:
    print(field, ":", getattr(omni_args, field, None))

[ ]
31 s

# Cell 14 — Run a forward/backward smoke test before full training

import torch

longest_idx = max(range(len(train_clean_ds)), key=lambda i: train_clean_ds[i]["input_length"])
example = train_clean_ds[longest_idx]

batch = data_collator([example])
batch = {k: v.to(device) for k, v in batch.items()}

model.train()
model.zero_grad(set_to_none=True)

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

with torch.autocast(
    device_type="cuda" if torch.cuda.is_available() else "cpu",
    dtype=torch.float16 if torch.cuda.is_available() else torch.bfloat16,
    enabled=torch.cuda.is_available(),
):
    outputs = model(**batch)
    smoke_loss = outputs.loss

print("Longest duration:", round(example["input_length"] / 16000, 2), "s")
print("Smoke-test loss:", float(smoke_loss))
smoke_loss.backward()

if torch.cuda.is_available():
    print("Peak GPU memory:", round(torch.cuda.max_memory_allocated() / 1024**3, 2), "GB")

print("Forward/backward successful.")
model.zero_grad(set_to_none=True)

[ ]
31 s

# Cell 15 — Create the controlled Fadhma fine-tuning configuration

from transformers import TrainingArguments

learning_rate = float(getattr(omni_args, "learning_rate", 1e-4))
train_batch_size = int(getattr(omni_args, "per_device_train_batch_size", 1))
eval_batch_size = int(getattr(omni_args, "per_device_eval_batch_size", 1))
grad_accum = int(getattr(omni_args, "gradient_accumulation_steps", 8))
epochs = float(getattr(omni_args, "num_train_epochs", 8))
warmup_steps = int(getattr(omni_args, "warmup_steps", 100))
weight_decay = float(getattr(omni_args, "weight_decay", 0.01))
seed = int(getattr(omni_args, "seed", 42))

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    per_device_train_batch_size=train_batch_size,
    per_device_eval_batch_size=eval_batch_size,
    gradient_accumulation_steps=grad_accum,
    learning_rate=learning_rate,
    weight_decay=weight_decay,
    warmup_steps=warmup_steps,
    num_train_epochs=epochs,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=25,
    load_best_model_at_end=True,
    metric_for_best_model="cer",
    greater_is_better=False,
    fp16=torch.cuda.is_available(),
    gradient_checkpointing=True,
    save_total_limit=2,
    train_sampling_strategy="group_by_length",
    length_column_name="input_length",
    remove_unused_columns=False,
    report_to="none",
    seed=seed,
)

print("Learning rate:", training_args.learning_rate)
print("Train batch size:", training_args.per_device_train_batch_size)
print("Eval batch size:", training_args.per_device_eval_batch_size)
print("Gradient accumulation:", training_args.gradient_accumulation_steps)
print("Epochs:", training_args.num_train_epochs)
print("Warmup steps:", training_args.warmup_steps)
print("Weight decay:", training_args.weight_decay)
print("Seed:", training_args.seed)

[ ]
31 s

# Cell 16 — Define a callback that saves the real best-CER model weights

from transformers import TrainerCallback
from pathlib import Path
import math

class BestCERBackupCallback(TrainerCallback):
    def __init__(self, save_dir, processor):
        self.save_dir = Path(save_dir)
        self.processor = processor
        self.best_cer = math.inf

    def on_evaluate(self, args, state, control, metrics=None, model=None, **kwargs):
        if metrics is None or model is None:
            return control

        current_cer = metrics.get("eval_cer")
        if current_cer is None:
            return control

        if current_cer < self.best_cer:
            self.best_cer = current_cer
            self.save_dir.mkdir(parents=True, exist_ok=True)
            model.save_pretrained(self.save_dir, safe_serialization=True)
            self.processor.save_pretrained(self.save_dir)

            with open(self.save_dir / "best_cer.txt", "w", encoding="utf-8") as f:
                f.write(f"best_cer={self.best_cer:.10f}\nepoch={state.epoch}\nglobal_step={state.global_step}\n")

            print(f"\n✓ Backed up new best model (CER={self.best_cer:.6f}, epoch={state.epoch})")

        return control

best_backup_callback = BestCERBackupCallback(BEST_BACKUP_DIR, processor)
print("Best-model backup directory:", BEST_BACKUP_DIR)

[ ]
31 s

# Cell 17 — Create the Hugging Face Trainer

from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_clean_ds,
    eval_dataset=val_clean_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[best_backup_callback],
)

print("Trainer ready.")
print("Training examples:", len(train_clean_ds))
print("Validation examples:", len(val_clean_ds))

[ ]
31 s

# Cell 18 — Start Fadhma-300M to Tarifit transfer fine-tuning

train_result = trainer.train()

[ ]
31 s

# Cell 19 — Save and verify the selected best model weights

trainer.save_model(str(FINAL_BEST_DIR))
processor.save_pretrained(str(FINAL_BEST_DIR))

print("Trainer best checkpoint:", trainer.state.best_model_checkpoint)
print("Trainer best metric:", trainer.state.best_metric)
print("Saved model:", FINAL_BEST_DIR)

weight_files = [
    p.name for p in FINAL_BEST_DIR.iterdir()
    if p.name in {"model.safetensors", "pytorch_model.bin", "model.safetensors.index.json", "pytorch_model.bin.index.json"}
]

print("Weight files:", weight_files)
assert weight_files, "ERROR: no model weight file was saved."
print("✓ Model weights verified on Drive.")

[ ]
31 s

# Cell 20 — Run greedy CTC inference with the best fine-tuned model

import torch
from tqdm.auto import tqdm

trainer.model.eval()
references = []
predictions = []

for example in tqdm(val_clean_ds, desc="Fadhma -> Tarifit validation"):
    input_values = torch.tensor(example["input_values"], dtype=torch.float32).unsqueeze(0).to(device)

    with torch.inference_mode():
        logits = trainer.model(input_values=input_values).logits

    pred_ids = torch.argmax(logits, dim=-1)
    prediction = processor.batch_decode(pred_ids)[0]
    reference = tokenizer.decode(example["labels"], group_tokens=False)

    predictions.append(eval_normalize(prediction))
    references.append(eval_normalize(reference))

print("Predictions:", len(predictions))

[ ]
31 s

# Cell 21 — Compute final WER and CER on the cleaned 128-segment validation set

from jiwer import wer, cer

final_wer = wer(references, predictions)
final_cer = cer(references, predictions)

print("=" * 70)
print("FADHMA-300M -> IMPROVED TARIFIT — CLEAN VALIDATION")
print("=" * 70)
print("Segments :", len(references))
print(f"WER      : {final_wer * 100:.2f}%")
print(f"CER      : {final_cer * 100:.2f}%")

[ ]
31 s

# Cell 22 — Save per-segment predictions and the experiment summary

import pandas as pd
from jiwer import wer, cer

results_df = pd.DataFrame({
    "clean_validation_index": range(len(val_clean_ds)),
    "original_validation_index": valid_val_indices,
    "duration_seconds": [x["input_length"] / 16000 for x in val_clean_ds],
    "reference": references,
    "prediction": predictions,
    "wer": [wer(r, p) for r, p in zip(references, predictions)],
    "cer": [cer(r, p) for r, p in zip(references, predictions)],
})

predictions_file = RESULTS_DIR / "fadhma_tarifit_v1_2_validation_128.csv"
results_df.to_csv(predictions_file, index=False, encoding="utf-8")

summary_df = pd.DataFrame([{
    "experiment": "Fadhma-300M -> improved Tarifit fine-tuning",
    "initial_model": FADHMA_ID,
    "training_dataset": str(IMPROVED_TRAIN_DIR),
    "train_examples": len(train_clean_ds),
    "validation_examples": len(val_clean_ds),
    "decoding": "greedy CTC",
    "language_model": False,
    "learning_rate": training_args.learning_rate,
    "train_batch_size": training_args.per_device_train_batch_size,
    "gradient_accumulation_steps": training_args.gradient_accumulation_steps,
    "epochs": training_args.num_train_epochs,
    "warmup_steps": training_args.warmup_steps,
    "weight_decay": training_args.weight_decay,
    "seed": training_args.seed,
    "wer_percent": final_wer * 100,
    "cer_percent": final_cer * 100,
    "best_checkpoint": trainer.state.best_model_checkpoint,
    "best_metric": trainer.state.best_metric,
}])

summary_file = RESULTS_DIR / "fadhma_tarifit_v1_2_summary.csv"
summary_df.to_csv(summary_file, index=False)

print("Saved predictions:", predictions_file)
print("Saved summary:", summary_file)
display(summary_df)

[ ]
31 s

# Cell 23 — Inspect the best and worst validation predictions by CER

display(
    results_df.sort_values("cer")
    [["original_validation_index", "duration_seconds", "reference", "prediction", "cer"]]
    .head(10)
)

display(
    results_df.sort_values("cer", ascending=False)
    [["original_validation_index", "duration_seconds", "reference", "prediction", "cer"]]
    .head(10)
)

Experiment conclusion

Complete this after the final WER/CER values are available.

Report the improved training corpus size/duration, final WER/CER, whether Kabyle pre-adaptation helped, and whether the improved transcription/alignment quality changed performance compared with earlier experiments.
Productos de pago de Colab
-
Cancelar contratos


# Keep the same clean validation set for comparability unless you later create a corrected validation version.
VALIDATION_DIR = PROJECT_ROOT / "data" / "processed" / "mms_corpus_v1_1" / "validation"

# Reuse the same 38-token Tarifit vocabulary if the alphabet is unchanged.
TARIFIT_VOCAB_PATH = PROJECT_ROOT / "data" / "processed" / "mms_tokenizer_v1_1" / "vocab.json"

# Previous OmniASR run: training_args.bin should still be present even though model weights were lost.
OMNI_REFERENCE_DIR = PROJECT_ROOT / "models" / "omniasr_w2v_300m_tarifit_v1"

OUTPUT_DIR = PROJECT_ROOT / "models" / "fadhma_300m_tarifit_v1_2"
BEST_BACKUP_DIR = PROJECT_ROOT / "models" / "fadhma_300m_tarifit_v1_2_best_backup"
FINAL_BEST_DIR = PROJECT_ROOT / "models" / "fadhma_300m_tarifit_v1_2_best"
RESULTS_DIR = PROJECT_ROOT / "results" / "fadhma_300m_tarifit_v1_2"

for path in [OUTPUT_DIR, BEST_BACKUP_DIR, FINAL_BEST_DIR, RESULTS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

FADHMA_ID = "agbalu/Fadhma-300M"
BAD_VAL_INDICES = {111, 118, 126, 127, 130}

print("Improved train directory:", IMPROVED_TRAIN_DIR)
print("Exists:", IMPROVED_TRAIN_DIR.exists())
print("Validation exists:", VALIDATION_DIR.exists())
print("Tokenizer exists:", TARIFIT_VOCAB_PATH.exists())

SyntaxError: invalid syntax (610227257.py, line 10)

In [ ]:
# Cell 4 — Load the improved training set and cleaned validation set

from datasets import load_from_disk

if not IMPROVED_TRAIN_DIR.exists():
    raise FileNotFoundError(
        "The improved training dataset was not found. Create/save it first, or change IMPROVED_TRAIN_DIR in Cell 3."
    )

train_ds = load_from_disk(str(IMPROVED_TRAIN_DIR))
val_ds = load_from_disk(str(VALIDATION_DIR))

valid_val_indices = [i for i in range(len(val_ds)) if i not in BAD_VAL_INDICES]
val_clean_ds = val_ds.select(valid_val_indices)

print("Improved training examples:", len(train_ds))
print("Improved training duration:", round(sum(x["input_length"] for x in train_ds) / 16000 / 3600, 2), "hours")
print("Original validation examples:", len(val_ds))
print("Clean validation examples:", len(val_clean_ds))

assert len(val_clean_ds) == 128

FileNotFoundError: The improved training dataset was not found. Create/save it first, or change IMPROVED_TRAIN_DIR in Cell 3.

In [ ]:
# Cell 4A — Check the available processed corpus folders

from pathlib import Path

PROCESSED_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
)

for path in sorted(PROCESSED_DIR.iterdir()):
    if path.is_dir():
        print(path.name)

forced_alignment
mms_M2_corpus_v1_1
mms_M2_tokenizer_v1_1
mms_corpus_v1_1
mms_tokenizer_v1_1
segments
tarifit_lm
transcripts_normalized
transcripts_raw
wav_16khz_mono
whisper_corpus_v1
xlsr_corpus_v1
xlsr_corpus_v1_1
xlsr_tokenizer
xlsr_tokenizer_v1_1


In [ ]:
# Cell 5 — Load the Tarifit tokenizer and inspect corrected references

from transformers import Wav2Vec2CTCTokenizer

tokenizer = Wav2Vec2CTCTokenizer(
    vocab_file=str(TARIFIT_VOCAB_PATH),
    unk_token="[UNK]",
    pad_token="[PAD]",
    word_delimiter_token="|"
)

print("Tarifit vocabulary size:", len(tokenizer))
print("PAD / CTC blank ID:", tokenizer.pad_token_id)

for i in range(min(5, len(train_ds))):
    ref = tokenizer.decode(train_ds[i]["labels"], group_tokens=False)
    print(f"{i}: {ref}")

In [ ]:
# Cell 6 — Inspect improved corpus statistics

import numpy as np

train_durations = np.array([x["input_length"] / 16000 for x in train_ds])
train_label_lengths = np.array([len(x["labels"]) for x in train_ds])

print("Audio duration — min/max/mean:", round(train_durations.min(),2), round(train_durations.max(),2), round(train_durations.mean(),2))
print("Label length — min/max/mean:", int(train_label_lengths.min()), int(train_label_lengths.max()), round(train_label_lengths.mean(),2))

In [ ]:
# Cell 7 — Build the Fadhma feature extractor with the Tarifit tokenizer

from transformers import AutoFeatureExtractor, Wav2Vec2Processor

feature_extractor = AutoFeatureExtractor.from_pretrained(FADHMA_ID)
processor = Wav2Vec2Processor(feature_extractor=feature_extractor, tokenizer=tokenizer)

print("Fadhma base model:", FADHMA_ID)
print("Sampling rate:", feature_extractor.sampling_rate)
print("Audio normalization:", feature_extractor.do_normalize)
print("Tarifit output vocabulary:", len(tokenizer))

In [ ]:
# Cell 8 — Load Fadhma-300M with a new Tarifit CTC output head

from transformers import Wav2Vec2ForCTC

model = Wav2Vec2ForCTC.from_pretrained(
    FADHMA_ID,
    vocab_size=len(tokenizer),
    pad_token_id=tokenizer.pad_token_id,
    ctc_loss_reduction="mean",
    ctc_zero_infinity=True,
    ignore_mismatched_sizes=True,
)

model.freeze_feature_encoder()
model.gradient_checkpointing_enable()
model.config.mask_time_prob = 0.05
model.config.layerdrop = 0.0
model = model.to(device)

print("Starting model:", FADHMA_ID)
print("Total parameters:", f"{model.num_parameters():,}")
print("Trainable parameters:", f"{sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
print("Tarifit output vocabulary:", model.config.vocab_size)
print("CTC head:", model.lm_head)

In [ ]:
# Cell 9 — Verify CTC feasibility for improved training and validation

import torch

def minimum_ctc_frames(labels):
    repeats = sum(labels[i] == labels[i-1] for i in range(1, len(labels)))
    return len(labels) + repeats

def check_ctc_feasibility(dataset, name):
    invalid = []
    for i, example in enumerate(dataset):
        output_frames = int(model._get_feat_extract_output_lengths(torch.tensor(example["input_length"])).item())
        minimum_frames = minimum_ctc_frames(example["labels"])
        if minimum_frames > output_frames:
            invalid.append({
                "index": i,
                "duration_seconds": example["input_length"] / 16000,
                "label_length": len(example["labels"]),
                "minimum_ctc_frames": minimum_frames,
                "available_output_frames": output_frames,
            })
    print(name, "— total:", len(dataset), "valid:", len(dataset)-len(invalid), "invalid:", len(invalid))
    return invalid

bad_train = check_ctc_feasibility(train_ds, "IMPROVED TRAIN")
bad_val = check_ctc_feasibility(val_clean_ds, "CLEAN VALIDATION")

if bad_train:
    print("First invalid training examples:")
    for item in bad_train[:10]:
        print(item)

assert len(bad_val) == 0

In [ ]:
# Cell 10 — Remove newly detected CTC-infeasible training examples if needed

if len(bad_train) == 0:
    train_clean_ds = train_ds
    print("No improved-training examples need removal.")
else:
    bad_train_indices = {x["index"] for x in bad_train}
    good_train_indices = [i for i in range(len(train_ds)) if i not in bad_train_indices]
    train_clean_ds = train_ds.select(good_train_indices)
    print("Removed:", len(bad_train_indices))
    print("Final training examples:", len(train_clean_ds))

In [ ]:
# Cell 11 — Create the dynamic-padding CTC data collator

from dataclasses import dataclass
from typing import Union

@dataclass
class DataCollatorCTCWithPadding:
    processor: any
    padding: Union[bool, str] = True

    def __call__(self, features):
        input_features = [{"input_values": x["input_values"]} for x in features]
        label_features = [{"input_ids": x["labels"]} for x in features]

        batch = self.processor.pad(input_features, padding=self.padding, return_tensors="pt")
        labels_batch = self.processor.tokenizer.pad(label_features, padding=self.padding, return_tensors="pt")

        batch["labels"] = labels_batch["input_ids"].masked_fill(
            labels_batch["attention_mask"].ne(1), -100
        )
        return batch

data_collator = DataCollatorCTCWithPadding(processor=processor)

sample_batch = data_collator([train_clean_ds[0], train_clean_ds[1]])
print("Input batch shape:", sample_batch["input_values"].shape)
print("Label batch shape:", sample_batch["labels"].shape)

In [ ]:
# Cell 12 — Define the common evaluation normalization and WER/CER metrics

import re
import unicodedata
import numpy as np
from jiwer import wer, cer

def eval_normalize(text):
    text = unicodedata.normalize("NFC", str(text))
    text = text.lower().strip()
    text = re.sub(r"\s+", " ", text)
    return text

def compute_metrics(pred):
    pred_ids = np.argmax(pred.predictions, axis=-1)
    label_ids = pred.label_ids.copy()
    label_ids[label_ids == -100] = tokenizer.pad_token_id

    pred_str = processor.batch_decode(pred_ids)
    label_str = tokenizer.batch_decode(label_ids, group_tokens=False)

    pred_str = [eval_normalize(x) for x in pred_str]
    label_str = [eval_normalize(x) for x in label_str]

    return {"wer": wer(label_str, pred_str), "cer": cer(label_str, pred_str)}

print("Metrics ready.")

In [ ]:
# Cell 13 — Recover the previous OmniASR optimization settings

import torch

training_args_files = sorted(OMNI_REFERENCE_DIR.glob("checkpoint-*/training_args.bin"))

if not training_args_files:
    raise FileNotFoundError("No OmniASR training_args.bin found. Check OMNI_REFERENCE_DIR in Cell 3.")

REFERENCE_ARGS_FILE = training_args_files[-1]
omni_args = torch.load(REFERENCE_ARGS_FILE, map_location="cpu", weights_only=False)

fields = [
    "learning_rate", "per_device_train_batch_size", "per_device_eval_batch_size",
    "gradient_accumulation_steps", "num_train_epochs", "warmup_steps",
    "weight_decay", "seed"
]

print("Reference args:", REFERENCE_ARGS_FILE)
for field in fields:
    print(field, ":", getattr(omni_args, field, None))

In [ ]:
# Cell 14 — Run a forward/backward smoke test before full training

import torch

longest_idx = max(range(len(train_clean_ds)), key=lambda i: train_clean_ds[i]["input_length"])
example = train_clean_ds[longest_idx]

batch = data_collator([example])
batch = {k: v.to(device) for k, v in batch.items()}

model.train()
model.zero_grad(set_to_none=True)

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

with torch.autocast(
    device_type="cuda" if torch.cuda.is_available() else "cpu",
    dtype=torch.float16 if torch.cuda.is_available() else torch.bfloat16,
    enabled=torch.cuda.is_available(),
):
    outputs = model(**batch)
    smoke_loss = outputs.loss

print("Longest duration:", round(example["input_length"] / 16000, 2), "s")
print("Smoke-test loss:", float(smoke_loss))
smoke_loss.backward()

if torch.cuda.is_available():
    print("Peak GPU memory:", round(torch.cuda.max_memory_allocated() / 1024**3, 2), "GB")

print("Forward/backward successful.")
model.zero_grad(set_to_none=True)

In [ ]:
# Cell 15 — Create the controlled Fadhma fine-tuning configuration

from transformers import TrainingArguments

learning_rate = float(getattr(omni_args, "learning_rate", 1e-4))
train_batch_size = int(getattr(omni_args, "per_device_train_batch_size", 1))
eval_batch_size = int(getattr(omni_args, "per_device_eval_batch_size", 1))
grad_accum = int(getattr(omni_args, "gradient_accumulation_steps", 8))
epochs = float(getattr(omni_args, "num_train_epochs", 8))
warmup_steps = int(getattr(omni_args, "warmup_steps", 100))
weight_decay = float(getattr(omni_args, "weight_decay", 0.01))
seed = int(getattr(omni_args, "seed", 42))

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    per_device_train_batch_size=train_batch_size,
    per_device_eval_batch_size=eval_batch_size,
    gradient_accumulation_steps=grad_accum,
    learning_rate=learning_rate,
    weight_decay=weight_decay,
    warmup_steps=warmup_steps,
    num_train_epochs=epochs,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=25,
    load_best_model_at_end=True,
    metric_for_best_model="cer",
    greater_is_better=False,
    fp16=torch.cuda.is_available(),
    gradient_checkpointing=True,
    save_total_limit=2,
    train_sampling_strategy="group_by_length",
    length_column_name="input_length",
    remove_unused_columns=False,
    report_to="none",
    seed=seed,
)

print("Learning rate:", training_args.learning_rate)
print("Train batch size:", training_args.per_device_train_batch_size)
print("Eval batch size:", training_args.per_device_eval_batch_size)
print("Gradient accumulation:", training_args.gradient_accumulation_steps)
print("Epochs:", training_args.num_train_epochs)
print("Warmup steps:", training_args.warmup_steps)
print("Weight decay:", training_args.weight_decay)
print("Seed:", training_args.seed)

In [ ]:
# Cell 16 — Define a callback that saves the real best-CER model weights

from transformers import TrainerCallback
from pathlib import Path
import math

class BestCERBackupCallback(TrainerCallback):
    def __init__(self, save_dir, processor):
        self.save_dir = Path(save_dir)
        self.processor = processor
        self.best_cer = math.inf

    def on_evaluate(self, args, state, control, metrics=None, model=None, **kwargs):
        if metrics is None or model is None:
            return control

        current_cer = metrics.get("eval_cer")
        if current_cer is None:
            return control

        if current_cer < self.best_cer:
            self.best_cer = current_cer
            self.save_dir.mkdir(parents=True, exist_ok=True)
            model.save_pretrained(self.save_dir, safe_serialization=True)
            self.processor.save_pretrained(self.save_dir)

            with open(self.save_dir / "best_cer.txt", "w", encoding="utf-8") as f:
                f.write(f"best_cer={self.best_cer:.10f}\nepoch={state.epoch}\nglobal_step={state.global_step}\n")

            print(f"\n✓ Backed up new best model (CER={self.best_cer:.6f}, epoch={state.epoch})")

        return control

best_backup_callback = BestCERBackupCallback(BEST_BACKUP_DIR, processor)
print("Best-model backup directory:", BEST_BACKUP_DIR)

In [ ]:
# Cell 17 — Create the Hugging Face Trainer

from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_clean_ds,
    eval_dataset=val_clean_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[best_backup_callback],
)

print("Trainer ready.")
print("Training examples:", len(train_clean_ds))
print("Validation examples:", len(val_clean_ds))

In [ ]:
# Cell 18 — Start Fadhma-300M to Tarifit transfer fine-tuning

train_result = trainer.train()

In [ ]:
# Cell 19 — Save and verify the selected best model weights

trainer.save_model(str(FINAL_BEST_DIR))
processor.save_pretrained(str(FINAL_BEST_DIR))

print("Trainer best checkpoint:", trainer.state.best_model_checkpoint)
print("Trainer best metric:", trainer.state.best_metric)
print("Saved model:", FINAL_BEST_DIR)

weight_files = [
    p.name for p in FINAL_BEST_DIR.iterdir()
    if p.name in {"model.safetensors", "pytorch_model.bin", "model.safetensors.index.json", "pytorch_model.bin.index.json"}
]

print("Weight files:", weight_files)
assert weight_files, "ERROR: no model weight file was saved."
print("✓ Model weights verified on Drive.")

In [ ]:
# Cell 20 — Run greedy CTC inference with the best fine-tuned model

import torch
from tqdm.auto import tqdm

trainer.model.eval()
references = []
predictions = []

for example in tqdm(val_clean_ds, desc="Fadhma -> Tarifit validation"):
    input_values = torch.tensor(example["input_values"], dtype=torch.float32).unsqueeze(0).to(device)

    with torch.inference_mode():
        logits = trainer.model(input_values=input_values).logits

    pred_ids = torch.argmax(logits, dim=-1)
    prediction = processor.batch_decode(pred_ids)[0]
    reference = tokenizer.decode(example["labels"], group_tokens=False)

    predictions.append(eval_normalize(prediction))
    references.append(eval_normalize(reference))

print("Predictions:", len(predictions))

In [ ]:
# Cell 21 — Compute final WER and CER on the cleaned 128-segment validation set

from jiwer import wer, cer

final_wer = wer(references, predictions)
final_cer = cer(references, predictions)

print("=" * 70)
print("FADHMA-300M -> IMPROVED TARIFIT — CLEAN VALIDATION")
print("=" * 70)
print("Segments :", len(references))
print(f"WER      : {final_wer * 100:.2f}%")
print(f"CER      : {final_cer * 100:.2f}%")

In [ ]:
# Cell 22 — Save per-segment predictions and the experiment summary

import pandas as pd
from jiwer import wer, cer

results_df = pd.DataFrame({
    "clean_validation_index": range(len(val_clean_ds)),
    "original_validation_index": valid_val_indices,
    "duration_seconds": [x["input_length"] / 16000 for x in val_clean_ds],
    "reference": references,
    "prediction": predictions,
    "wer": [wer(r, p) for r, p in zip(references, predictions)],
    "cer": [cer(r, p) for r, p in zip(references, predictions)],
})

predictions_file = RESULTS_DIR / "fadhma_tarifit_v1_2_validation_128.csv"
results_df.to_csv(predictions_file, index=False, encoding="utf-8")

summary_df = pd.DataFrame([{
    "experiment": "Fadhma-300M -> improved Tarifit fine-tuning",
    "initial_model": FADHMA_ID,
    "training_dataset": str(IMPROVED_TRAIN_DIR),
    "train_examples": len(train_clean_ds),
    "validation_examples": len(val_clean_ds),
    "decoding": "greedy CTC",
    "language_model": False,
    "learning_rate": training_args.learning_rate,
    "train_batch_size": training_args.per_device_train_batch_size,
    "gradient_accumulation_steps": training_args.gradient_accumulation_steps,
    "epochs": training_args.num_train_epochs,
    "warmup_steps": training_args.warmup_steps,
    "weight_decay": training_args.weight_decay,
    "seed": training_args.seed,
    "wer_percent": final_wer * 100,
    "cer_percent": final_cer * 100,
    "best_checkpoint": trainer.state.best_model_checkpoint,
    "best_metric": trainer.state.best_metric,
}])

summary_file = RESULTS_DIR / "fadhma_tarifit_v1_2_summary.csv"
summary_df.to_csv(summary_file, index=False)

print("Saved predictions:", predictions_file)
print("Saved summary:", summary_file)
display(summary_df)

In [ ]:
# Cell 23 — Inspect the best and worst validation predictions by CER

display(
    results_df.sort_values("cer")
    [["original_validation_index", "duration_seconds", "reference", "prediction", "cer"]]
    .head(10)
)

display(
    results_df.sort_values("cer", ascending=False)
    [["original_validation_index", "duration_seconds", "reference", "prediction", "cer"]]
    .head(10)
)

## Experiment conclusion

_Complete this after the final WER/CER values are available._

Report the improved training corpus size/duration, final WER/CER, whether Kabyle pre-adaptation helped, and whether the improved transcription/alignment quality changed performance compared with earlier experiments.